# PARSER CODE FOR PROJECT (Krisha.kz вторичный и первичный)

In [8]:
!pip install requests beautifulsoup4 pandas lxml --quiet

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

BASE_URL = "https://krisha.kz/prodazha/kvartiry/almaty/"
PARAMS = {
    "das[who]": "1"
}

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

MAX_PAGES = 100
DELAY = 1

all_rows = []


# --- цена ---
def clean_price(text):
    if not text:
        return None
    text = text.replace('\xa0', '').replace('₸', '')
    return int(re.sub(r'\D', '', text)) if re.sub(r'\D', '', text) else None


# --- ПАРСИНГ TITLE  ---
def parse_title(title):
    rooms = None
    area = None
    district = None

    if not title:
        return rooms, area, district

    text = title.lower()

    # комнаты
    m = re.search(r'(\d+)[-\s]?ком', text)
    if m:
        rooms = int(m.group(1))

    # площадь
    m = re.search(r'([\d.]+)\s*м²', text)
    if m:
        area = float(m.group(1))

    # район (примерно)
    m = re.search(r'([а-яё\s\-]+район)', text)
    if m:
        district = m.group(1).strip()

    return rooms, area, district


def parse_page(page):
    PARAMS["page"] = page

    response = requests.get(BASE_URL, headers=HEADERS, params=PARAMS)

    if response.status_code != 200:
        print("Error:", response.status_code)
        return []

    soup = BeautifulSoup(response.text, "lxml")

    cards = soup.select("div.a-card")

    results = []

    for card in cards:
        # Цена
        price_el = card.select_one(".a-card__price")
        price = clean_price(price_el.text.strip()) if price_el else None

        # Title
        title_el = card.select_one(".a-card__title")
        title_text = title_el.text.strip() if title_el else None

        # Link
        link = "https://krisha.kz" + title_el["href"] if title_el else None

        # ПАРСИНГ ИЗ TITLE
        rooms, area, district = parse_title(title_text)

        # fallback (если вдруг нет площади)
        if not area:
            info_el = card.select_one(".a-card__subtitle")
            if info_el:
                m = re.search(r'([\d.]+)\s*м²', info_el.text)
                if m:
                    area = float(m.group(1))

        # описание
        desc_el = card.select_one(".a-card__text-preview")
        description = desc_el.text.strip() if desc_el else None

        results.append({
            "price": price,
            "rooms": rooms,
            "area_m2": area,
            "district": district,
            "title": title_text,
            "description": description,
            "link": link,
            "page": page
        })

    return results


# --- сбор ---
for page in range(1, MAX_PAGES + 1):
    print("Page:", page)

    data = parse_page(page)

    if not data:
        break

    all_rows.extend(data)

    time.sleep(DELAY)


Page: 1
Page: 2
Page: 3
Page: 4
Page: 5
Page: 6
Page: 7
Page: 8
Page: 9
Page: 10
Page: 11
Page: 12
Page: 13
Page: 14
Page: 15
Page: 16
Page: 17
Page: 18
Page: 19
Page: 20
Page: 21
Page: 22
Page: 23
Page: 24
Page: 25
Page: 26
Page: 27
Page: 28
Page: 29
Page: 30
Page: 31
Page: 32
Page: 33
Page: 34
Page: 35
Page: 36
Page: 37
Page: 38
Page: 39
Page: 40
Page: 41
Page: 42
Page: 43
Page: 44
Page: 45
Page: 46
Page: 47
Page: 48
Page: 49
Page: 50
Page: 51
Page: 52
Page: 53
Page: 54
Page: 55
Page: 56
Page: 57
Page: 58
Page: 59
Page: 60
Page: 61
Page: 62
Page: 63
Page: 64
Page: 65
Page: 66
Page: 67
Page: 68
Page: 69
Page: 70
Page: 71
Page: 72
Page: 73
Page: 74
Page: 75
Page: 76
Page: 77
Page: 78
Page: 79
Page: 80
Page: 81
Page: 82
Page: 83
Page: 84
Page: 85
Page: 86
Page: 87
Page: 88
Page: 89
Page: 90
Page: 91
Page: 92
Page: 93
Page: 94
Page: 95
Page: 96
Page: 97
Page: 98
Page: 99
Page: 100


In [9]:
# --- dataframe ---
df = pd.DataFrame(all_rows)

df["price_per_m2"] = df["price"] / df["area_m2"]

df.to_csv("krisha_secondary_almaty_FIXED.csv", index=False)

print("DONE:", len(df))
df.info()
df.head()

DONE: 2000
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   price         2000 non-null   int64  
 1   rooms         2000 non-null   int64  
 2   area_m2       2000 non-null   float64
 3   district      0 non-null      object 
 4   title         2000 non-null   object 
 5   description   2000 non-null   object 
 6   link          2000 non-null   object 
 7   page          2000 non-null   int64  
 8   price_per_m2  2000 non-null   float64
dtypes: float64(2), int64(3), object(4)
memory usage: 140.8+ KB


,price,rooms,area_m2,district,title,description,link,page,price_per_m2
0,155000000,4,153.00,None,4-комнатная квартира · 153 м² · 8/9 этаж,"монолитный дом, 2003 г.п., потолки 3м., санузе...",https://krisha.kz/a/show/761623800,1,1.013072e+06
1,106000000,3,150.00,None,3-комнатная квартира · 150 м² · 4/10 этаж,"жил. комплекс Коктем по Тимирязева-Шашкина, мо...",https://krisha.kz/a/show/1009843904,1,7.066667e+05
2,70587000,4,102.30,None,4-комнатная квартира · 102.3 м²,"жил. комплекс Dream City. Art, 6 этажей, 2026 ...",https://krisha.kz/a/show/1009995074,1,6.900000e+05
3,45647000,3,65.21,None,3-комнатная квартира · 65.21 м²,"жил. комплекс Maxima City, 9 этажей, 2025 г.п....",https://krisha.kz/a/show/688265083,1,7.000000e+05
4,57742809,3,81.98,None,3-комнатная квартира · 81.98 м²,"жил. комплекс Dream City. Eco, 6 этажей, 2025 ...",https://krisha.kz/a/show/694706344,1,7.043524e+05


In [12]:
import re

def extract_district(text):
    if not text:
        return None

    text = text.lower()

    districts = [
        "алмалинский район",
        "ауэзовский район",
        "бостандыкский район",
        "медеуский район",
        "наурызбайский район",
        "турксибский район",
        "жетысуский район"
    ]

    for d in districts:
        if d in text:
            return d

    return None


df["district"] = df["description"].apply(extract_district)
df["district"] = df["title"].apply(extract_district)

mask = df["district"].isna()
df.loc[mask, "district"] = df.loc[mask, "description"].apply(extract_district)

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   price         2000 non-null   int64  
 1   rooms         2000 non-null   int64  
 2   area_m2       2000 non-null   float64
 3   district      20 non-null     object 
 4   title         2000 non-null   object 
 5   description   2000 non-null   object 
 6   link          2000 non-null   object 
 7   page          2000 non-null   int64  
 8   price_per_m2  2000 non-null   float64
dtypes: float64(2), int64(3), object(4)
memory usage: 140.8+ KB


In [16]:
print(df.head(10))

       price  rooms  area_m2 district  \
0  155000000      4   153.00     None   
1  106000000      3   150.00     None   
2   70587000      4   102.30     None   
3   45647000      3    65.21     None   
4   57742809      3    81.98     None   
5   75000000      2    66.00     None   
6   44569500      2    64.50     None   
7   48490000      3    74.60     None   
8   30000000      2    65.50     None   
9   48560160      2    64.65     None   

                                       title  \
0   4-комнатная квартира · 153 м² · 8/9 этаж   
1  3-комнатная квартира · 150 м² · 4/10 этаж   
2            4-комнатная квартира · 102.3 м²   
3            3-комнатная квартира · 65.21 м²   
4            3-комнатная квартира · 81.98 м²   
5    2-комнатная квартира · 66 м² · 6/9 этаж   
6             2-комнатная квартира · 64.5 м²   
7             3-комнатная квартира · 74.6 м²   
8             2-комнатная квартира · 65.5 м²   
9            2-комнатная квартира · 64.65 м²   

                   

In [22]:
df["price_segment"] = pd.qcut(df["price_per_m2"], 3, labels=["cheap", "medium", "expensive"])

In [29]:
df.to_csv("krisha_secondary_almaty_cleaned.csv", index=False)

In [30]:
# parser первичный рынок

BASE_URL = "https://krisha.kz/prodazha/kvartiry/almaty/"
PARAMS = {
    "das[novostroiki]": "1"   # первичный рынок
}

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

MAX_PAGES = 100
DELAY = 1

all_rows = []


# --- цена ---
def clean_price(text):
    if not text:
        return None
    text = text.replace('\xa0', '').replace('₸', '')
    return int(re.sub(r'\D', '', text)) if re.sub(r'\D', '', text) else None


# --- ПАРСИНГ TITLE  ---
def parse_title(title):
    rooms = None
    area = None
    district = None

    if not title:
        return rooms, area, district

    text = title.lower()

    # комнаты
    m = re.search(r'(\d+)[-\s]?ком', text)
    if m:
        rooms = int(m.group(1))

    # площадь
    m = re.search(r'([\d.]+)\s*м²', text)
    if m:
        area = float(m.group(1))

    # район (примерно)
    m = re.search(r'([а-яё\s\-]+район)', text)
    if m:
        district = m.group(1).strip()

    return rooms, area, district


def parse_page(page):
    PARAMS["page"] = page

    response = requests.get(BASE_URL, headers=HEADERS, params=PARAMS)

    if response.status_code != 200:
        print("Error:", response.status_code)
        return []

    soup = BeautifulSoup(response.text, "lxml")

    cards = soup.select("div.a-card")

    results = []

    for card in cards:
        # Цена
        price_el = card.select_one(".a-card__price")
        price = clean_price(price_el.text.strip()) if price_el else None

        # Title
        title_el = card.select_one(".a-card__title")
        title_text = title_el.text.strip() if title_el else None

        # Link
        link = "https://krisha.kz" + title_el["href"] if title_el else None

        # ПАРСИНГ ИЗ TITLE
        rooms, area, district = parse_title(title_text)

        # fallback (если вдруг нет площади)
        if not area:
            info_el = card.select_one(".a-card__subtitle")
            if info_el:
                m = re.search(r'([\d.]+)\s*м²', info_el.text)
                if m:
                    area = float(m.group(1))

        # описание
        desc_el = card.select_one(".a-card__text-preview")
        description = desc_el.text.strip() if desc_el else None

        results.append({
            "price": price,
            "rooms": rooms,
            "area_m2": area,
            "district": district,
            "title": title_text,
            "description": description,
            "link": link,
            "page": page
        })

    return results


# --- сбор ---
for page in range(1, MAX_PAGES + 1):
    print("Page:", page)

    data = parse_page(page)

    if not data:
        break

    all_rows.extend(data)

    time.sleep(DELAY)


Page: 1
Page: 2
Page: 3
Page: 4
Page: 5
Page: 6
Page: 7
Page: 8
Page: 9


In [32]:
# --- dataframe ---
df_first_krisha = pd.DataFrame(all_rows)

df_first_krisha["price_per_m2"] = df_first_krisha["price"] / df_first_krisha["area_m2"]

df_first_krisha.to_csv("krisha_primary_almaty_FIXED.csv", index=False)

print("DONE:", len(df_first_krisha))
df_first_krisha.info()
df_first_krisha.head()

DONE: 146
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 146 entries, 0 to 145
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   price         146 non-null    int64  
 1   rooms         146 non-null    int64  
 2   area_m2       146 non-null    float64
 3   district      0 non-null      object 
 4   title         146 non-null    object 
 5   description   146 non-null    object 
 6   link          146 non-null    object 
 7   page          146 non-null    int64  
 8   price_per_m2  146 non-null    float64
dtypes: float64(2), int64(3), object(4)
memory usage: 10.4+ KB


,price,rooms,area_m2,district,title,description,link,page,price_per_m2
0,17787000,1,36.30,None,1-комнатная квартира · 36.3 м²,"жил. комплекс Dostar, 9 этажей, 2026 г.п., пот...",https://krisha.kz/a/show/673332516,1,4.900000e+05
1,17600250,1,37.85,None,1-комнатная квартира · 37.85 м²,"жил. комплекс Keruen, 6 этажей, 2025 г.п., пот...",https://krisha.kz/a/show/1000459786,1,4.650000e+05
2,67440000,1,58.30,None,1-комнатная квартира · 58.3 м²,"жил. комплекс Etasa Residence, 18 этажей, 2023...",https://krisha.kz/a/show/684354905,1,1.156775e+06
3,128359000,2,77.00,None,2-комнатная квартира · 77 м²,"жил. комплекс Diamond, 9 этажей, 2028 г.п., по...",https://krisha.kz/a/show/1008497882,1,1.667000e+06
4,29216000,2,53.12,None,2-комнатная квартира · 53.12 м²,"жил. комплекс Аккент, 12 этажей, 2023 г.п., по...",https://krisha.kz/a/show/1008768026,1,5.500000e+05


In [34]:
df_first_krisha["price_segment"] = pd.qcut(df_first_krisha["price_per_m2"], 3, labels=["cheap", "medium", "expensive"])

df_first_krisha.to_csv("krisha_primary_almaty_cleaned.csv", index=False)

df_first_krisha.info()
df_first_krisha.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 146 entries, 0 to 145
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   price          146 non-null    int64   
 1   rooms          146 non-null    int64   
 2   area_m2        146 non-null    float64 
 3   district       0 non-null      object  
 4   title          146 non-null    object  
 5   description    146 non-null    object  
 6   link           146 non-null    object  
 7   page           146 non-null    int64   
 8   price_per_m2   146 non-null    float64 
 9   price_segment  146 non-null    category
dtypes: category(1), float64(2), int64(3), object(4)
memory usage: 10.7+ KB


,price,rooms,area_m2,district,title,description,link,page,price_per_m2,price_segment
0,17787000,1,36.30,None,1-комнатная квартира · 36.3 м²,"жил. комплекс Dostar, 9 этажей, 2026 г.п., пот...",https://krisha.kz/a/show/673332516,1,4.900000e+05,cheap
1,17600250,1,37.85,None,1-комнатная квартира · 37.85 м²,"жил. комплекс Keruen, 6 этажей, 2025 г.п., пот...",https://krisha.kz/a/show/1000459786,1,4.650000e+05,cheap
2,67440000,1,58.30,None,1-комнатная квартира · 58.3 м²,"жил. комплекс Etasa Residence, 18 этажей, 2023...",https://krisha.kz/a/show/684354905,1,1.156775e+06,expensive
3,128359000,2,77.00,None,2-комнатная квартира · 77 м²,"жил. комплекс Diamond, 9 этажей, 2028 г.п., по...",https://krisha.kz/a/show/1008497882,1,1.667000e+06,expensive
4,29216000,2,53.12,None,2-комнатная квартира · 53.12 м²,"жил. комплекс Аккент, 12 этажей, 2023 г.п., по...",https://krisha.kz/a/show/1008768026,1,5.500000e+05,cheap
